In [1]:
import pandas as pd
import yfinance as yf
import time
import os
import re
from IPython.display import clear_output

In [2]:
#ADJ is the amount in dollars to make up for the price diff bw/ yfinance api and broker data
yfinance_sym_dic = { 
    'MNQ': {'SYM':'MNQ=F', 'ADJ': 0},
    'NQ': {'SYM':'NQ=F', 'ADJ': 0},
    'US100': {'SYM':'MNQ=F', 'ADJ': -53.18},
    'GC': {'SYM':'GC=F', 'ADJ': 0},
    'MGX': {'SYM':'MGC=F', 'ADJ': 0},
    'SI': {'SYM':'SI=F', 'ADJ': 0},
    'SIL': {'SYM':'SIL=F', 'ADJ': 0},
    'XAUUSD': {'SYM':'GC=F', 'ADJ': 0},
    'AGXUSD': {'SYM':'SI=F', 'ADJ': 0},
    'BZ': {'SYM':'BZ=F', 'ADJ': 0}, # Brent Crude Futures
    'CL': {'SYM':'CL=F', 'ADJ': 0}, # WTI Crude Futures
    'BTC': {'SYM':'BTC-USD', 'ADJ': 0},
    'ETH': {'SYM':'ETH-USD', 'ADJ': 0}
}


def get_live_price(ticker_symbol: str, yfinance_map: dict)-> float:
    # Initialize the Ticker object 
    if ticker_symbol in yfinance_map.keys():
        ticker = yf.Ticker(yfinance_map[ticker_symbol]['SYM'])
        # .fast_info provides the most recent 'last_price'
        # This is faster than fetching the full .info dictionary
        current_price = ticker.fast_info['last_price'] + yfinance_map[ticker_symbol]['ADJ']
    else:
        ticker = yf.Ticker(ticker_symbol)
        current_price = ticker.fast_info['last_price']
        
    return current_price

# Read the trades worksheet

In [3]:
# 1. Replace with your actual Google Sheet ID
# (Found in the URL: https://docs.google.com/spreadsheets/d/SHEET_ID/edit)
SHEET_ID = "1HJ9h7UEtUQCXNA58UkZyPsHogJWBAcB1lNWt9nOPMR4"

# 2. Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Trades'

# 3. Construct the export URL
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"

# 4. Load into DataFrame
df = pd.read_csv(url)

# Cast numeric columns from str to float type
cols = ['Open Price', 'Close Price', 'Commission']
for c in cols:
    df[c] = df[c].apply(lambda x: float(re.sub(r"\(", "-", re.sub(r"[,\)]", "", x))))
df.head()

,Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Risk ($),PnL,Closed,Close Date,Entry Link,Exit Link 1,Exit Link 2
0,8/24/2026,Paper Trading #1,COF,-34.0,221.02,216.24,0.00,"59,800.00",NaN,162.52,No,NaN,Link,NaN,NaN
1,8/24/2026,Tradestation - Equity,COF,-34.0,220.62,216.24,0.00,NaN,NaN,148.92,No,NaN,Link,NaN,NaN
2,8/24/2026,Tradestation - Equity,CVNA,-130.0,72.08,81.08,-2.91,"42,336.39","(1,172.61)","(1,172.61)",No,NaN,Link,NaN,NaN
3,8/25/2026,Tradestation - Futures,MNQ,-2.0,29343.00,29480.75,0.00,"13,015.34",NaN,(551.00),No,NaN,Link,NaN,NaN
4,8/25/2026,FTP - 147759,US100,-1.1,29276.90,29408.05,0.00,"99,385.65",(144.26),(144.26),No,NaN,Link,NaN,NaN


# Get Point Values

In [4]:
# 2. Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Symbols'

# 3. Construct the export URL
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"

# 4. Load into DataFrame
point_val_df = pd.read_csv(url, header=None, names=['Symbol', 'Point Value'])
point_val_df.head()

,Symbol,Point Value
0,COF,1
1,CVNA,1
2,MNQ,2
3,QQQ,1
4,UKOIL,1


# Get Prices

In [5]:
price_df = pd.DataFrame(df['Symbol']).drop_duplicates()
price_df['Current Price'] = price_df.Symbol.apply(lambda x : get_live_price(x, yfinance_sym_dic))
price_df

,Symbol,Current Price
0,COF,216.240005
2,CVNA,75.769997
3,MNQ,29287.000000
4,US100,29233.820000


# Append Price to trades DF

In [6]:
df = pd.merge(df, price_df, on='Symbol', how='left')
df = pd.merge(df, point_val_df, on='Symbol', how='left')
df['Point Value'] = df['Point Value'].fillna(1)
df['PnL'] = (df['Volume'] * (df['Current Price']-df['Open Price']) * df['Point Value']).round(2)
df

,Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Risk ($),PnL,Closed,Close Date,Entry Link,Exit Link 1,Exit Link 2,Current Price,Point Value
0,8/24/2026,Paper Trading #1,COF,-34.0,221.02,216.24,0.00,"59,800.00",NaN,162.52,No,NaN,Link,NaN,NaN,216.240005,1
1,8/24/2026,Tradestation - Equity,COF,-34.0,220.62,216.24,0.00,NaN,NaN,148.92,No,NaN,Link,NaN,NaN,216.240005,1
2,8/24/2026,Tradestation - Equity,CVNA,-130.0,72.08,81.08,-2.91,"42,336.39","(1,172.61)",-479.70,No,NaN,Link,NaN,NaN,75.769997,1
3,8/25/2026,Tradestation - Futures,MNQ,-2.0,29343.00,29480.75,0.00,"13,015.34",NaN,224.00,No,NaN,Link,NaN,NaN,29287.000000,2
4,8/25/2026,FTP - 147759,US100,-1.1,29276.90,29408.05,0.00,"99,385.65",(144.26),47.39,No,NaN,Link,NaN,NaN,29233.820000,1
5,8/25/2026,FTP - 147759,US100,-6.0,29170.87,29260.45,0.00,"99,385.65",(537.48),-377.70,No,NaN,Link,NaN,NaN,29233.820000,1
6,8/25/2026,FTP - 147759,US100,-1.6,29215.65,29246.51,0.00,"99,385.65",(49.38),-29.07,No,NaN,Link,NaN,NaN,29233.820000,1
7,8/25/2026,Paper Trading #1,CVNA,-171.0,75.57,82.50,0.00,"59,313.65","(1,185.03)",-34.20,No,NaN,NaN,NaN,NaN,75.769997,1


# Group by account and symbol to report

In [7]:
out = df.groupby(['Symbol','Account']).agg({'PnL': sum})
print(out)

                                  PnL
Symbol Account                       
COF    Paper Trading #1        162.52
       Tradestation - Equity   148.92
CVNA   Paper Trading #1        -34.20
       Tradestation - Equity  -479.70
MNQ    Tradestation - Futures  224.00
US100  FTP - 147759           -359.38
